### layer1 preprocessing

In [3]:
# Loading in the json file 1 (layer1.json)
import json
with open('layer1.json' , 'r') as file:
  data_1 = json.load(file)

In [ ]:
# unique ingredient strings from layer1.json
unique_ingredients = sorted({ing.strip().lower()
                             for recipe in data_1
                             for ing in (i.get("text") for i in recipe["ingredients"])
                             if ing})
len(unique_ingredients), unique_ingredients[:25]

In [ ]:
len(data_1)

In [ ]:
type(data_1)

In [ ]:
type(data_1[0])

In [ ]:
data_1[0].keys()

In [4]:
data_1[0]

{'ingredients': [{'text': '6 ounces penne'},
  {'text': '2 cups Beechers Flagship Cheese Sauce (recipe follows)'},
  {'text': '1 ounce Cheddar, grated (1/4 cup)'},
  {'text': '1 ounce Gruyere cheese, grated (1/4 cup)'},
  {'text': '1/4 to 1/2 teaspoon chipotle chili powder (see Note)'},
  {'text': '1/4 cup (1/2 stick) unsalted butter'},
  {'text': '1/3 cup all-purpose flour'},
  {'text': '3 cups milk'},
  {'text': '14 ounces semihard cheese (page 23), grated (about 3 1/2 cups)'},
  {'text': '2 ounces semisoft cheese (page 23), grated (1/2 cup)'},
  {'text': '1/2 teaspoon kosher salt'},
  {'text': '1/4 to 1/2 teaspoon chipotle chili powder'},
  {'text': '1/8 teaspoon garlic powder'},
  {'text': '(makes about 4 cups)'}],
 'url': 'http://www.epicurious.com/recipes/food/views/-world-s-best-mac-and-cheese-387747',
 'partition': 'train',
 'title': 'Worlds Best Mac and Cheese',
 'id': '000018c8a5',
 'instructions': [{'text': 'Preheat the oven to 350 F. Butter or oil an 8-inch baking dish.'},


In [ ]:
type(data_1[0]['id'])

In [ ]:
next((row for row in data_1 if row.get("id") == "000018c8a5"), None)

In [ ]:
next((row for row in data_1 if row.get("title") == "Minestrone Soup With Ground Beef Recipe"), None)

In [ ]:
# Storing all the values of (id, title , partition) in a list 
import pandas as pd
from tqdm import tqdm 

id = []
title = []
part = []
url = [] 

# Looping through and appending the values to a newly created list 
for attr in tqdm(data_1):
  id.append(attr['id'])
  title.append(attr['title'])
  part.append(attr['partition'])
  url.append(attr['url'])

In [ ]:
# Building a dictionary with above values 
data_1_dict = {'ID': id , 'food_title': title , 'partition': part , "image_url": url}

# Convert this beautiful dict into a fully fledge dataframe 
data1_df = pd.DataFrame(data_1_dict)

In [ ]:
# Looking first 15 samples 
data1_df.head(15)

In [ ]:
data1_df.to_csv("layer1.csv")

In [ ]:
def get_list_of_strings(key , json_file):
    all_ingredient_lists = []
    for recipie in tqdm(json_file):
        ingredients = recipie[key]
        ingredient_list = []
        for ingredient in ingredients:
            ingredient_list.append(ingredient['text'])
            
        all_ingredient_lists.append(ingredient_list)
    return all_ingredient_lists


# Using the above function and parsing out the values
ingredients_list = get_list_of_strings('ingredients' , data_1)
instructions_list = get_list_of_strings('instructions' , data_1)

In [ ]:
# Creating updated list with one individual strings 
updated_ingredients_list = [' /t '.join(ingredients_list[i]) for i in tqdm(range(0 , len(data_1)))] 
updated_instructions_list = [' /t '.join(instructions_list[i]) for i in tqdm(range(0 , len(data_1)))]

In [ ]:
import numpy as np 

data1_df['ingredients'] = pd.Series(updated_ingredients_list)
data1_df['instructions'] = pd.Series(updated_instructions_list)

data1_df.head()

In [ ]:
data1_df.shape

In [ ]:
data1_df.to_csv("layer1.csv", index = False)

### recipe1m-ex-limited.json (Canonical Ingredients)

In [6]:
import json

def iter_json_array(path, chunk_size=1 << 20):
    decoder = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        buf = ""
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                return
            buf += chunk
            idx = buf.find("[")
            if idx != -1:
                buf = buf[idx + 1:]
                break
        while True:
            buf = buf.lstrip()
            if buf.startswith("]"):
                return
            try:
                obj, end = decoder.raw_decode(buf)
                yield obj
                buf = buf[end:].lstrip()
                if buf.startswith(","):
                    buf = buf[1:]
            except json.JSONDecodeError:
                chunk = f.read(chunk_size)
                if not chunk:
                    return
                buf += chunk

limited_path = "../foodkg.github.io/src/verify/data/recipe1m-ex-limited.json"
layer1_path = "../foodkg.github.io/src/prep-scripts/in/layer1.json"

limited_set = set()
for rec in iter_json_array(limited_path):
    for ing in rec["ingredients"]:
        name = ing.get("name")
        if name:
            limited_set.add(name.strip().lower())

layer1_set = set()
for rec in iter_json_array(layer1_path):
    for ing in rec["ingredients"]:
        text = ing.get("text")
        if text:
            layer1_set.add(text.strip().lower())

print("unique limited:", len(limited_set))
print("unique layer1:", len(layer1_set))
print("limited/layer1 ratio:", len(limited_set) / max(1, len(layer1_set)))


unique limited: 15851
unique layer1: 2317227
limited/layer1 ratio: 0.00684050375729266


In [8]:
import json

def iter_json_array(path, chunk_size=1 << 20):
    decoder = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        buf = ""
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                return
            buf += chunk
            idx = buf.find("[")
            if idx != -1:
                buf = buf[idx + 1:]
                break
        while True:
            buf = buf.lstrip()
            if buf.startswith("]"):
                return
            try:
                obj, end = decoder.raw_decode(buf)
                yield obj
                buf = buf[end:].lstrip()
                if buf.startswith(","):
                    buf = buf[1:]
            except json.JSONDecodeError:
                chunk = f.read(chunk_size)
                if not chunk:
                    return
                buf += chunk

limited_path = "../foodkg.github.io/src/verify/data/recipe1m-ex-limited.json"
layer1_path = "../foodkg.github.io/src/prep-scripts/in/layer1.json"

# build a small lookup from limited (first N recipes)
N = 5
limited_lookup = {}
for rec in iter_json_array(limited_path):
    limited_lookup[rec["id"]] = [ing["name"] for ing in rec["ingredients"] if ing.get("name")]
    if len(limited_lookup) >= N:
        break

# stream layer1 and print matched pairs
printed = 0
for rec in iter_json_array(layer1_path):
    if rec["id"] in limited_lookup:
        raw_texts = [ing["text"] for ing in rec["ingredients"] if ing.get("text")]
        print(f"\nID: {rec['id']}")
        print("Layer1 texts:", raw_texts)
        print("Normalized:", limited_lookup[rec["id"]])
        printed += 1
        if printed >= N:
            break



ID: 000018c8a5
Layer1 texts: ['6 ounces penne', '2 cups Beechers Flagship Cheese Sauce (recipe follows)', '1 ounce Cheddar, grated (1/4 cup)', '1 ounce Gruyere cheese, grated (1/4 cup)', '1/4 to 1/2 teaspoon chipotle chili powder (see Note)', '1/4 cup (1/2 stick) unsalted butter', '1/3 cup all-purpose flour', '3 cups milk', '14 ounces semihard cheese (page 23), grated (about 3 1/2 cups)', '2 ounces semisoft cheese (page 23), grated (1/2 cup)', '1/2 teaspoon kosher salt', '1/4 to 1/2 teaspoon chipotle chili powder', '1/8 teaspoon garlic powder', '(makes about 4 cups)']
Normalized: ['penne', 'cheese sauce', 'cheddar cheese', 'gruyere cheese', 'dried chipotle powder', 'unsalted butter', 'all - purpose flour', 'milk', 'kosher salt', 'dried chipotle powder', 'garlic powder']

ID: 000033e39b
Layer1 texts: ['1 c. elbow macaroni', '1 c. cubed American cheese (4 ounce.)', '1/2 c. sliced celery', '1/2 c. minced green pepper', '3 tbsp. minced pimento', '1/2 c. mayonnaise or possibly salad dressi

In [5]:
# Loading in the json file 1 (layer1.json)
import json
with open('../foodkg.github.io/src/prep-scripts/out/recipe1m-ex-limited.json' , 'r') as file:
  data_limited = json.load(file)

In [7]:
data_limited[22]

{'id': '00015b5a39',
 'title': "Erin's Mashed Potatoes",
 'url': 'http://www.food.com/recipe/erins-mashed-potatoes-26236',
 'ingredients': [{'unit': '', 'quantity': '4', 'name': 'potatoes'},
  {'unit': 'cup', 'quantity': '1/2', 'name': 'red wine'},
  {'unit': 'cup', 'quantity': '1/2', 'name': 'teriyaki sauce'},
  {'unit': 'tablespoons', 'quantity': '2', 'name': 'garlic'},
  {'unit': 'cup', 'quantity': '1/2', 'name': 'sour cream'}],
 'instructions': ['Peel the potatoes and quarter.',
  'Add all ingredients to a medium saucepan and simmer until the potatoes are tender.',
  'Remove potatoes from cooking liquid and mash to desired consistencey (I like mine a bit lumpy).',
  'Add a few spoonfuls of the cooking liquid and the sour cream and combine.',
  'Serve as a side with steak!'],
 'tags': []}

In [ ]:
data_limited[22].keys()

In [9]:
len(data_limited)

1029720

In [ ]:
next((row for row in data_limited if row.get("data_limited") == "000035f7ed"), None)

In [10]:
unique_ingredients = sorted({ing["name"].strip().lower()
                             for recipe in data_limited
                             for ing in recipe["ingredients"]
                             if ing.get("name")})
len(unique_ingredients), unique_ingredients[:25]


(18252,
 ['1% fat buttermilk',
  '1% fat cottage cheese',
  '1% low - fat chocolate milk',
  '1% low - fat milk',
  '10 - inch corn tortillas',
  '10 - inch flour tortilla',
  '10 - inch flour tortillas',
  '10 - inch pie shell',
  '10 - inch tart shell',
  '10 - inch unbaked pastry shell',
  '10 - minute herb stuffing mix',
  '10 - minute success rice',
  '10% cream',
  '100 - calorie honey maid cinnamon graham cracker crisps',
  '100 - calorie snack pack tapioca pudding',
  '100 proof vodka',
  '100% bran',
  '100% natural raisin date cereal',
  '12 - inch flour tortilla',
  '12 - inch flour tortillas',
  '12 - inch pizza crust',
  '13 bean soup mix',
  '15 bean mix',
  '15 bean soup mix',
  '15% cream'])

### det_ingrs

In [ ]:
# Loading in the json file 1 (layer1.json)
import json
with open('../foodkg.github.io/src/prep-scripts/in/det_ingrs.json' , 'r') as file:
  data_ingr = json.load(file)

In [ ]:
next((row for row in data_ingr if row.get("id") == "00015b5a39"), None)

### Recipes with nutritional info

In [ ]:
# Loading in the json file 1 (layer1.json)
import json
with open('../foodkg.github.io/src/prep-scripts/out/recipes_with_nutritional_info.json' , 'r') as file:
  data_nutr_info = json.load(file)

In [ ]:
len(data_nutr_info)

In [ ]:
data_nutr_info[0]

In [ ]:
data_nutr_info[0].keys()

### layer2+

In [ ]:
# Loading in the json file 1 (layer1.json)
import json
with open('../foodkg.github.io/src/prep-scripts/in/layer2+.json' , 'r') as file:
  data_2 = json.load(file)

In [ ]:
len(data_2)

In [ ]:
type(data_2)

In [ ]:
len(data_2[4].get("images"))

In [ ]:
data_2

In [ ]:
next((row for row in data_2 if row.get("id") == "000095fc1d"), None)

In [ ]:
# data2 = list of layer2 records
rec = next((r for r in data_2 if r["id"] == "000095fc1d"), None)
if rec and rec["images"]:
    photo_url = rec["images"][0]["url"]  # pick first image
    photo_id = rec["images"][0]["id"]
    print(photo_id, photo_url)


In [ ]:
photos = [img["url"] for img in rec["images"]] if rec else []


In [ ]:
photos